***Step 1: Tạo SparkSession và load các dataset cần thiết***

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IntrusionDetection") \
    .master("local[2]") \
    .getOrCreate()

spark

#Tập training
df_training = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/Training and Testing Sets/UNSW_NB15_training-set.csv",
    header=True, 
    inferSchema=True)
df_training.count()

#

175341

**-Tập ground truth**: Chứa các thông tin về các cuộc tấn công

In [2]:
df_gt = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/NUSW-NB15_GT.csv",
    header=True,
    inferSchema=True)
df_gt.show(5)
df_gt.columns

+----------+----------+---------------+-------------------+--------+------------+-----------+--------------+----------------+--------------------+--------------------+---+
|Start time| Last time|Attack category| Attack subcategory|Protocol|   Source IP|Source Port|Destination IP|Destination Port|         Attack Name|    Attack Reference|  .|
+----------+----------+---------------+-------------------+--------+------------+-----------+--------------+----------------+--------------------+--------------------+---+
|1421927414|1421927416| Reconnaissance|               HTTP|     tcp|175.45.176.0|      13284|149.171.126.16|              80|Domino Web Server...|                   -|  .|
|1421927415|1421927415|       Exploits|   Unix 'r' Service|     udp|175.45.176.3|      21223|149.171.126.18|           32780|Solaris rwalld Fo...|CVE 2002-0573 (ht...|  .|
|1421927416|1421927416|       Exploits|            Browser|     tcp|175.45.176.2|      23357|149.171.126.16|              80|Windows Metafil

['Start time',
 'Last time',
 'Attack category',
 'Attack subcategory',
 'Protocol',
 'Source IP',
 'Source Port',
 'Destination IP',
 'Destination Port',
 'Attack Name',
 'Attack Reference',
 '.']

**-Đọc các features của datasets:** Bao gồm 49 features khác nhau cho các lượt truy cập mạng

In [3]:
import pandas as pd
features = pd.read_csv("/home/jovyan/intrusion_data/raw/CSV Files/NUSW-NB15_features.csv", encoding="cp1252")
col_names = list(features["Name"])
print(col_names)
print(len(col_names))

['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']
49


**-Đọc các bản ghi gốc và kiểm tra các cột:**

In [31]:
raw_df = spark.read.csv(
    "/home/jovyan/intrusion_data/raw/CSV Files/UNSW-NB15_1.csv",
    header=True,
    inferSchema=True,
    schema=None
).toDF(*col_names)

raw_df.select("srcip", "dstip", "Stime", "attack_cat").show(5)
# see the shape
print(f"Rows: {raw_df.count()}")
print(f"Columns: {len(raw_df.columns)}")

+----------+-------------+----------+----------+
|     srcip|        dstip|     Stime|attack_cat|
+----------+-------------+----------+----------+
|59.166.0.0|149.171.126.9|1421927414|      NULL|
|59.166.0.6|149.171.126.7|1421927414|      NULL|
|59.166.0.5|149.171.126.5|1421927414|      NULL|
|59.166.0.3|149.171.126.0|1421927414|      NULL|
|59.166.0.0|149.171.126.9|1421927414|      NULL|
+----------+-------------+----------+----------+
only showing top 5 rows

Rows: 700000
Columns: 49


In [32]:
raw_df_2 = spark.read.csv(
    "/home/jovyan/intrusion_data/raw/CSV Files/UNSW-NB15_2.csv",
    header=True,
    inferSchema=True,
    schema=None
).toDF(*col_names)
raw_df_2.select("srcip", "dstip", "Stime", "attack_cat").show(5)
# see the shape
print(f"Rows: {raw_df.count()}")
print(f"Columns: {len(raw_df.columns)}")

+----------+-------------+----------+----------+
|     srcip|        dstip|     Stime|attack_cat|
+----------+-------------+----------+----------+
|59.166.0.0|149.171.126.3|1421955842|      NULL|
|59.166.0.8|149.171.126.6|1421955842|      NULL|
|59.166.0.0|149.171.126.3|1421955842|      NULL|
|59.166.0.8|149.171.126.6|1421955842|      NULL|
|59.166.0.8|149.171.126.3|1421955842|      NULL|
+----------+-------------+----------+----------+
only showing top 5 rows

Rows: 700000
Columns: 49


In [33]:
raw_df_3 = spark.read.csv(
    "/home/jovyan/intrusion_data/raw/CSV Files/UNSW-NB15_2.csv",
    header=True,
    inferSchema=True,
    schema=None
).toDF(*col_names)
raw_df_3.select("srcip", "dstip", "Stime", "attack_cat").show(5)
# see the shape
print(f"Rows: {raw_df.count()}")
print(f"Columns: {len(raw_df.columns)}")

+----------+-------------+----------+----------+
|     srcip|        dstip|     Stime|attack_cat|
+----------+-------------+----------+----------+
|59.166.0.0|149.171.126.3|1421955842|      NULL|
|59.166.0.8|149.171.126.6|1421955842|      NULL|
|59.166.0.0|149.171.126.3|1421955842|      NULL|
|59.166.0.8|149.171.126.6|1421955842|      NULL|
|59.166.0.8|149.171.126.3|1421955842|      NULL|
+----------+-------------+----------+----------+
only showing top 5 rows

Rows: 700000
Columns: 49


**Kiểm tra list events:**


In [5]:
list_event_df = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/UNSW-NB15_LIST_EVENTS.csv",
    header=True,
    inferSchema=True,
    schema=None)

list_event_df.show()

+----------------+--------------------+----------------+
| Attack category|  Attack subcategory|Number of events|
+----------------+--------------------+----------------+
|          normal|                NULL|         2218761|
|        Fuzzers |                 FTP|             558|
|        Fuzzers |                HTTP|            1497|
|        Fuzzers |                 RIP|            3550|
|        Fuzzers |                 SMB|            5245|
|        Fuzzers |              Syslog|            1851|
|        Fuzzers |                PPTP|            1583|
|         Fuzzers|                 FTP|             248|
|         Fuzzers|              DCERPC|             164|
|         Fuzzers|                OSPF|             993|
|        Fuzzers |                TFTP|             193|
|        Fuzzers |             DCERPC |             455|
|        Fuzzers |                OSPF|            1746|
|        Fuzzers |                 BGP|            6163|
| Reconnaissance |             

**Step 2: Load các công cụ cần thiết để khám phá dữ liệu:**

In [6]:
from pyspark.sql.functions import from_unixtime, col,to_timestamp, isnan, when, count, window

In [34]:
raw_df = raw_df.union(raw_df_2).union(raw_df_3)

In [35]:

raw_df = raw_df.withColumn(
    "timestamp", 
    from_unixtime(col("Stime"))
)

raw_df.select("srcip", "timestamp", "attack_cat").show(5)
raw_df = raw_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

raw_df.printSchema()

+----------+-------------------+----------+
|     srcip|          timestamp|attack_cat|
+----------+-------------------+----------+
|59.166.0.0|2015-01-22 11:50:14|      NULL|
|59.166.0.6|2015-01-22 11:50:14|      NULL|
|59.166.0.5|2015-01-22 11:50:14|      NULL|
|59.166.0.3|2015-01-22 11:50:14|      NULL|
|59.166.0.0|2015-01-22 11:50:14|      NULL|
+----------+-------------------+----------+
only showing top 5 rows

root
 |-- srcip: string (nullable = true)
 |-- sport: string (nullable = true)
 |-- dstip: string (nullable = true)
 |-- dsport: string (nullable = true)
 |-- proto: string (nullable = true)
 |-- state: string (nullable = true)
 |-- dur: double (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- sttl: integer (nullable = true)
 |-- dttl: integer (nullable = true)
 |-- sloss: integer (nullable = true)
 |-- dloss: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- Sload: double (nullable = true)
 |-- Dload: 

In [37]:

raw_df.groupBy(
    window("timestamp", "5 minutes")
).agg(
    count("*").alias("packet_count")
).orderBy("window").show()

+--------------------+------------+
|              window|packet_count|
+--------------------+------------+
|{2015-01-22 11:45...|           6|
|{2015-01-22 11:50...|        7382|
|{2015-01-22 11:55...|        7424|
|{2015-01-22 12:00...|        8712|
|{2015-01-22 12:05...|        7805|
|{2015-01-22 12:10...|        7575|
|{2015-01-22 12:15...|        6571|
|{2015-01-22 12:20...|        6796|
|{2015-01-22 12:25...|        7506|
|{2015-01-22 12:30...|        7619|
|{2015-01-22 12:35...|        7495|
|{2015-01-22 12:40...|        8597|
|{2015-01-22 12:45...|       10855|
|{2015-01-22 12:50...|        8355|
|{2015-01-22 12:55...|        8403|
|{2015-01-22 13:00...|        8156|
|{2015-01-22 13:05...|        9137|
|{2015-01-22 13:10...|        8699|
|{2015-01-22 13:15...|        8165|
|{2015-01-22 13:20...|        8047|
+--------------------+------------+
only showing top 20 rows



In [38]:

gt_renamed = df_gt.withColumnRenamed("Source IP", "srcip") \
                  .withColumnRenamed("Source Port", "sport") \
                  .withColumnRenamed("Destination IP", "dstip") \
                  .withColumnRenamed("Destination Port", "dsport")\
                  .withColumnRenamed("Protocol", "proto") \
                  .withColumnRenamed("Start time", "Stime")\
                  .withColumnRenamed("Last time", "Ltime")

In [14]:
gt_renamed.printSchema()

root
 |-- Stime: string (nullable = true)
 |-- Ltime: string (nullable = true)
 |-- Attack category: string (nullable = true)
 |-- Attack subcategory: string (nullable = true)
 |-- proto: string (nullable = true)
 |-- srcip: string (nullable = true)
 |-- sport: integer (nullable = true)
 |-- dstip: string (nullable = true)
 |-- dsport: integer (nullable = true)
 |-- Attack Name: string (nullable = true)
 |-- Attack Reference: string (nullable = true)
 |-- .: string (nullable = true)



In [15]:
gt_renamed.show()

+----------+----------+---------------+--------------------+-----+------------+-----+--------------+------+--------------------+--------------------+---+
|     Stime|     Ltime|Attack category|  Attack subcategory|proto|       srcip|sport|         dstip|dsport|         Attack Name|    Attack Reference|  .|
+----------+----------+---------------+--------------------+-----+------------+-----+--------------+------+--------------------+--------------------+---+
|1421927414|1421927416| Reconnaissance|                HTTP|  tcp|175.45.176.0|13284|149.171.126.16|    80|Domino Web Server...|                   -|  .|
|1421927415|1421927415|       Exploits|    Unix 'r' Service|  udp|175.45.176.3|21223|149.171.126.18| 32780|Solaris rwalld Fo...|CVE 2002-0573 (ht...|  .|
|1421927416|1421927416|       Exploits|             Browser|  tcp|175.45.176.2|23357|149.171.126.16|    80|Windows Metafile ...|CVE 2005-4560 (ht...|  .|
|1421927417|1421927417|       Exploits| Miscellaneous Batch|  tcp|175.45.176

In [39]:
gt_renamed = gt_renamed.withColumn("Stime",col("Stime").cast('integer'))
gt_renamed = gt_renamed.withColumn("Ltime", col("Ltime").cast('integer'))
gt_renamed.printSchema()

root
 |-- Stime: integer (nullable = true)
 |-- Ltime: integer (nullable = true)
 |-- Attack category: string (nullable = true)
 |-- Attack subcategory: string (nullable = true)
 |-- proto: string (nullable = true)
 |-- srcip: string (nullable = true)
 |-- sport: integer (nullable = true)
 |-- dstip: string (nullable = true)
 |-- dsport: integer (nullable = true)
 |-- Attack Name: string (nullable = true)
 |-- Attack Reference: string (nullable = true)
 |-- .: string (nullable = true)



In [43]:
df_raw = raw_df.join(gt_renamed, on= ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'Stime', 'Ltime'], how='left')

In [41]:
df_raw.count()

3606581

In [26]:
raw_df.count()

700000

In [51]:
df_1 = raw_df_2.filter(col('attack_cat').isNotNull())
df_1_1 = df_1.groupBy("srcip").count()
df_1_1.show()

+------------+-----+
|       srcip|count|
+------------+-----+
|175.45.176.1| 4418|
|175.45.176.3|18501|
|175.45.176.0|25881|
|175.45.176.2| 3949|
+------------+-----+



In [45]:
raw_df.show()

+------------+-----+--------------+------+-----+-----+--------+------+------+----+----+-----+-----+-------+------------+-----------+-----+-----+----+----+-----+-----+-------+-------+-----------+-----------+---------+---------+----------+----------+--------+--------+------+------+------+---------------+------------+----------------+------------+----------+----------+----------+----------+-----------+----------------+----------------+--------------+----------+-----+-------------------+
|       srcip|sport|         dstip|dsport|proto|state|     dur|sbytes|dbytes|sttl|dttl|sloss|dloss|service|       Sload|      Dload|Spkts|Dpkts|swin|dwin|stcpb|dtcpb|smeansz|dmeansz|trans_depth|res_bdy_len|     Sjit|     Djit|     Stime|     Ltime| Sintpkt| Dintpkt|tcprtt|synack|ackdat|is_sm_ips_ports|ct_state_ttl|ct_flw_http_mthd|is_ftp_login|ct_ftp_cmd|ct_srv_src|ct_srv_dst|ct_dst_ltm|ct_src_ ltm|ct_src_dport_ltm|ct_dst_sport_ltm|ct_dst_src_ltm|attack_cat|Label|          timestamp|
+------------+-----+--